In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_parquet(
    "../data/raw/gios_pm_measurements_1y.parquet"
)

# 1. Kształt danych

In [ ]:
df.head()

In [ ]:
df.info()

### Wnioski

Zbiór zawiera 225 014 rekordów i 9 kolumn.  
Nie występują braki typu `NaN` w istniejących rekordach.  
Kolumna `Data` ma poprawny typ `datetime64[ns]`, dzięki czemu dane są gotowe do analizy szeregów czasowych.

In [ ]:
print("Od:", df["Data"].min())
print("Do:", df["Data"].max())

In [ ]:
df["Parametr"].value_counts()

In [ ]:
df["station_id"].nunique()

In [ ]:
df["sensor_id"].nunique()

In [ ]:
stations = (
    df[
        [
            "station_id",
            "station_name",
            "city",
            "voivodeship",
        ]
    ]
    .drop_duplicates()
    .sort_values("voivodeship")
)

stations

In [ ]:
df.isna().sum()

### Brakujące wartości

W zbiorze nie występują wartości typu NaN.  
Oznacza to, że każdy istniejący rekord posiada komplet podstawowych informacji.

Braki mogą jednak występować jako całkowicie brakujące momenty czasowe,
dlatego konieczna jest osobna analiza luk w szeregu czasowym.

In [ ]:
df = df.sort_values(
    ["sensor_id", "Data"]
).copy()

In [ ]:
df["time_diff"] = (
    df.groupby("sensor_id")["Data"]
    .diff()
)

In [ ]:
df["time_diff"].value_counts().head(10)

In [ ]:
sensor_frequency = (
    df.groupby("sensor_id")["time_diff"]
    .agg(
        lambda x: (
            x.mode().iloc[0]
            if not x.mode().empty
            else pd.NaT
        )
    )
)

In [ ]:
sensor_frequency.value_counts()

In [ ]:
df["expected_diff"] = (
    df["sensor_id"].map(sensor_frequency)
)

In [ ]:
df["is_gap"] = (
    df["time_diff"] > df["expected_diff"]
)

In [ ]:
df["is_gap"].sum()

In [ ]:
df["missing_measurements"] = (
    df["time_diff"] / df["expected_diff"] - 1
).clip(lower=0)

In [ ]:
df["missing_measurements"].sum()

In [ ]:
gap_summary = (
    df.groupby(
        [
            "station_id",
            "station_name",
            "Parametr",
        ]
    )
    .agg(
        observed_measurements=("Wartość", "count"),
        missing_measurements=(
            "missing_measurements",
            "sum",
        ),
    )
    .reset_index()
)

In [ ]:
gap_summary["expected_measurements"] = (
    gap_summary["observed_measurements"]
    + gap_summary["missing_measurements"]
)

In [ ]:
gap_summary["missing_percent"] = (
    gap_summary["missing_measurements"]
    / gap_summary["expected_measurements"]
    * 100
)

In [ ]:
gap_summary.sort_values(
    "missing_percent",
    ascending=False,
)

In [ ]:
sensor_ranges = (
    df.groupby(
        [
            "station_id",
            "station_name",
            "sensor_id",
            "Parametr",
        ]
    )
    .agg(
        first_measurement=("Data", "min"),
        last_measurement=("Data", "max"),
        measurements=("Data", "count"),
    )
    .reset_index()
)

In [ ]:
sensor_ranges["frequency"] = (
    sensor_ranges["sensor_id"].map(sensor_frequency)
)

In [ ]:
sensor_ranges.sort_values(
    ["station_name", "Parametr"]
)

### Analiza częstotliwości i luk czasowych

W zbiorze znajduje się 43 sensory. Dla 26 sensorów dominującą
częstotliwością pomiarów jest 1 godzina, natomiast dla 17 sensorów
1 doba.

Wykryto 515 przerw dłuższych niż typowa częstotliwość danego sensora.
Przerwy te odpowiadają łącznie 4373 brakującym pomiarom wewnątrz
obserwowanych szeregów czasowych.

Samo występowanie przerwy nie oznacza wartości równej zero – brakujący
pomiar oznacza brak obserwacji.

W dalszej analizie pokrycia należy dodatkowo uwzględnić początek
i koniec dostępności każdego sensora względem pełnego analizowanego
zakresu dat.

In [ ]:
pm10 = df[
    df["Parametr"] == "PM10"
]["Wartość"].dropna()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    pm10,
    bins=50,
)

plt.xlabel("PM10 [µg/m³]")
plt.ylabel("Liczba pomiarów")
plt.title("Rozkład stężeń PM10")

plt.tight_layout()
plt.show()

In [ ]:
pm25 = df[
    df["Parametr"] == "PM2.5"
]["Wartość"].dropna()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    pm25,
    bins=50,
)

plt.xlabel("PM2.5 [µg/m³]")
plt.ylabel("Liczba pomiarów")
plt.title("Rozkład stężeń PM2.5")

plt.tight_layout()
plt.show()

In [ ]:
df.groupby("Parametr")["Wartość"].describe()

### Wnioski z rozkładów PM

Dla obu parametrów średnia jest wyższa od mediany, co wskazuje
na prawostronną skośność rozkładu.

PM10 osiąga wyższe wartości przeciętne niż PM2.5.
W zbiorze występują również wysokie wartości maksymalne,
jednak nie są one automatycznie traktowane jako błędne obserwacje,
ponieważ mogą odpowiadać rzeczywistym epizodom smogowym,
szczególnie w sezonie zimowym.

In [ ]:
katowice_pm10 = df[
    (df["city"] == "Katowice")
    & (df["Parametr"] == "PM10")
].copy()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    katowice_pm10["Data"],
    katowice_pm10["Wartość"],
)

plt.xlabel("Data")
plt.ylabel("PM10 [µg/m³]")
plt.title("Szereg czasowy PM10 - Katowice")

plt.tight_layout()
plt.show()

In [ ]:
katowice_pm25 = df[
    (df["city"] == "Katowice")
    & (df["Parametr"] == "PM2.5")
].copy()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    katowice_pm25["Data"],
    katowice_pm25["Wartość"],
)

plt.xlabel("Data")
plt.ylabel("PM2.5 [µg/m³]")
plt.title("Szereg czasowy PM2.5 - Katowice")

plt.tight_layout()
plt.show()

### Wnioski z analizy szeregów czasowych

Dla PM10 i PM2.5 widoczna jest wyraźna sezonowość.
Najwyższe wartości pojawiają się głównie w okresie zimowym,
natomiast w miesiącach letnich stężenia są niższe i bardziej stabilne.

Zaobserwowane wysokie zimowe piki nie są automatycznie traktowane
jako błędy lub wartości odstające, ponieważ mogą odpowiadać
rzeczywistym epizodom smogowym.

### raport pokrycia + wstepny wybor stacji do modelu

In [ ]:
analysis_start = pd.Timestamp("2025-08-20 00:00")
analysis_end = pd.Timestamp("2026-08-20 23:00")

In [ ]:
sensor_ranges["expected_full_period"] = (
    (analysis_end - analysis_start)
    / sensor_ranges["frequency"]
    + 1
)

In [ ]:
sensor_ranges["coverage_percent"] = (
    sensor_ranges["measurements"]
    / sensor_ranges["expected_full_period"]
    * 100
)

In [ ]:
sensor_ranges["missing_percent_full_period"] = (
    100 - sensor_ranges["coverage_percent"]
)

In [ ]:
sensor_ranges.sort_values(
    "coverage_percent"
).head(15)

In [ ]:
sensor_ranges.sort_values(
    "coverage_percent",
    ascending=False
).head(15)

In [ ]:
coverage_report = sensor_ranges[
    [
        "station_id",
        "station_name",
        "sensor_id",
        "Parametr",
        "frequency",
        "first_measurement",
        "last_measurement",
        "measurements",
        "expected_full_period",
        "coverage_percent",
        "missing_percent_full_period",
    ]
].copy()

In [ ]:
coverage_report.to_csv(
    "../data/processed/gios_coverage_report.csv",
    index=False,
)

### Wstępny wybór stacji do modelu

Na podstawie raportu pokrycia stwierdzono, że 12 z 16 analizowanych stacji
posiada godzinowe pomiary zarówno PM10, jak i PM2.5.

Najwyższe pokrycie danych występuje m.in. dla stacji w Łodzi, Gorzowie
Wielkopolskim, Krakowie, Gdańsku, Katowicach, Elblągu i Poznaniu,
gdzie dostępność danych dla obu parametrów przekracza około 98–99%.

Stacje we Wrocławiu, Lublinie, Opolu i Rzeszowie nie posiadają
godzinowych danych dla obu analizowanych parametrów, dlatego są
mniej odpowiednie do modelu opartego na danych godzinowych.

Ostateczny wybór stacji zostanie wykonany po połączeniu danych
o jakości powietrza z danymi pogodowymi i sprawdzeniu wspólnego
pokrycia czasowego.

# 2. Statystyka

## 2.1. Sezonowość PM

In [ ]:
df["month"] = df["Data"].dt.month

In [ ]:
monthly_pm = (
    df.groupby(["month", "Parametr"])["Wartość"]
    .mean()
    .reset_index()
)

monthly_pm

In [ ]:
plt.figure(figsize=(10, 5))

for parameter in ["PM10", "PM2.5"]:
    data = monthly_pm[
        monthly_pm["Parametr"] == parameter
    ]

    plt.plot(
        data["month"],
        data["Wartość"],
        marker="o",
        label=parameter,
    )

plt.xlabel("Miesiąc")
plt.ylabel("Średnie stężenie [µg/m³]")
plt.title("Średnie stężenie PM według miesiąca")
plt.xticks(range(1, 13))
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def get_season(month):
    if month in [10, 11, 12, 1, 2, 3]:
        return "heating"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "transition"

In [ ]:
df["season"] = df["month"].apply(get_season)

In [ ]:
season_pm = (
    df.groupby(["season", "Parametr"])["Wartość"]
    .mean()
    .reset_index()
)

season_pm

### Wnioski z analizy sezonowości

Średnie stężenia PM10 i PM2.5 są wyraźnie wyższe w sezonie grzewczym
niż w okresie letnim.

Dla PM10 średnia w sezonie grzewczym wynosi około 29.45 µg/m³,
natomiast latem około 14.69 µg/m³.

Dla PM2.5 średnia w sezonie grzewczym wynosi około 22.52 µg/m³,
a latem około 8.67 µg/m³.

Wyniki wskazują na silną sezonowość stężeń pyłów zawieszonych,
z najwyższymi wartościami w miesiącach zimowych.

## 2.2. Dzień roboczy vs weekend

In [ ]:
df["day_of_week"] = df["Data"].dt.dayofweek

In [ ]:
df["is_weekend"] = df["day_of_week"] >= 5

In [ ]:
df[["Data", "day_of_week", "is_weekend"]].head()

In [ ]:
weekday_weekend_summary = (
    df.groupby(["Parametr", "is_weekend"])["Wartość"]
    .agg(["count", "mean", "median", "std"])
)

weekday_weekend_summary

In [ ]:
print(
    "PM10 różnica średnich:",
    pm10_weekend_mean - pm10_weekday_mean,
)

print(
    "PM2.5 różnica średnich:",
    pm25_weekend_mean - pm25_weekday_mean,
)

In [ ]:
from scipy.stats import mannwhitneyu

In [ ]:
pm10_weekday = df[
    (df["Parametr"] == "PM10")
    & (~df["is_weekend"])
]["Wartość"].dropna()

pm10_weekend = df[
    (df["Parametr"] == "PM10")
    & (df["is_weekend"])
]["Wartość"].dropna()

In [ ]:
u_stat_pm10, p_value_pm10 = mannwhitneyu(
    pm10_weekday,
    pm10_weekend,
    alternative="two-sided",
)

print("PM10")
print("U =", u_stat_pm10)
print("p-value =", p_value_pm10)

In [ ]:
pm25_weekday = df[
    (df["Parametr"] == "PM2.5")
    & (~df["is_weekend"])
]["Wartość"].dropna()

pm25_weekend = df[
    (df["Parametr"] == "PM2.5")
    & (df["is_weekend"])
]["Wartość"].dropna()

In [ ]:
u_stat_pm25, p_value_pm25 = mannwhitneyu(
    pm25_weekday,
    pm25_weekend,
    alternative="two-sided",
)

print("PM2.5")
print("U =", u_stat_pm25)
print("p-value =", p_value_pm25)

### Dzień roboczy a weekend

Średnie stężenie PM10 wyniosło 23.77 µg/m³ w dni robocze
oraz 20.87 µg/m³ w weekendy. Różnica średnich wyniosła
około 2.90 µg/m³.

Dla PM2.5 średnie wartości wyniosły odpowiednio 16.55 µg/m³
i 15.16 µg/m³, co daje różnicę około 1.40 µg/m³.

Do porównania rozkładów zastosowano nieparametryczny test
Manna–Whitneya. Otrzymano p = 1.54 × 10⁻¹¹⁰ dla PM10
oraz p = 3.13 × 10⁻¹⁵ dla PM2.5.

### Interpretacja testu Manna–Whitneya

Dla PM10 otrzymano p-value = 1,54 × 10⁻¹¹⁰, a dla PM2.5
p-value = 3,13 × 10⁻¹⁵. Obie wartości są znacznie mniejsze
od przyjętego poziomu istotności α = 0,05.

Oznacza to, że istnieją statystycznie istotne różnice między
rozkładami stężeń PM w dni robocze i w weekendy.

Średnie stężenie PM10 było w weekendy niższe o około 2,90 µg/m³,
a PM2.5 niższe o około 1,40 µg/m³ w porównaniu z dniami roboczymi.

Wyniki sugerują więc, że w analizowanym zbiorze stężenia pyłów
zawieszonych były niższe w weekendy niż w dni robocze.

## 2.3. Zależność PM od pogody

In [ ]:
weather_df = pd.read_parquet(
    "../data/raw/open_meteo_weather_1y.parquet"
)

In [ ]:
weather_df.head()

In [ ]:
weather_df.info()

In [ ]:
from datetime import timezone, timedelta

In [ ]:
cet = timezone(timedelta(hours=1))

In [ ]:
df["timestamp_local"] = (
    df["Data"]
    .dt.tz_localize(cet)
)

In [ ]:
df["timestamp_utc"] = (
    df["timestamp_local"]
    .dt.tz_convert("UTC")
)

In [ ]:
df[
    [
        "Data",
        "timestamp_local",
        "timestamp_utc",
    ]
].head()

In [ ]:
merged_df = df.merge(
    weather_df,
    on=["station_id", "timestamp_utc"],
    how="left",
)

In [ ]:
merged_df[
    [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]
].isna().mean() * 100

### korelacje

In [ ]:
pm10_corr = merged_df[
    merged_df["Parametr"] == "PM10"
][
    [
        "Wartość",
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]
].corr()

pm10_corr

In [ ]:
pm25_corr = merged_df[
    merged_df["Parametr"] == "PM2.5"
][
    [
        "Wartość",
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]
].corr()

pm25_corr

In [ ]:
merged_df["temperature_2m"].isna().sum()

### Korelacja PM z warunkami pogodowymi

Dla PM10 stwierdzono ujemną korelację z temperaturą
(r = -0,355) oraz z prędkością wiatru (r = -0,243).
Korelacja z wilgotnością była bardzo słaba i dodatnia
(r = 0,097).

Dla PM2.5 najsilniejszą zależność zaobserwowano dla temperatury
(r = -0,482). Korelacja z prędkością wiatru również była ujemna
(r = -0,229), natomiast z wilgotnością dodatnia
(r = 0,269).

Wyniki wskazują, że wyższa temperatura i silniejszy wiatr
są związane z niższymi stężeniami pyłów zawieszonych.
Zależność ta jest szczególnie widoczna dla temperatury i PM2.5.